In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

### The Network Name in db and Ted's spread

In [4]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl

df["Station ID"] = pd.to_numeric(df["Station ID"], errors="coerce").astype("Int64")
df["Network ID"] = pd.to_numeric(df["Network ID"], errors="coerce").astype("Int64")


wanted_issues = [
    "Concatenate"
]

df_concat = df[
    df["ISSUE"].str.strip().isin(wanted_issues) &
    (df["Network ID"] != 11)
]

df_concat

df_concat_new =  df_concat[['Network ID','Station ID', 'Unique Names', 'Native ID']].reset_index(drop=True)

df_concat_new

,Network ID,Station ID,Unique Names,Native ID
0,9,12939,-> Fort St John Key Learning Centre,Fort St John Key Learning Centre -> 469
1,9,12938,-> Pine River Gas Plant,Pine River Gas Plant -> 107
2,9,2640,Harmac Pacific,M102038 -> 60 -> 294
3,9,2623,Kelowna College,M112070 -> 102
4,9,12531,NaN,Birchbank Golf Course -?> 97
5,9,12534,NaN,Burns Lake Fire Centre -?> 104
6,9,12535,NaN,Burns Lake Sheraton East -?> 439
7,9,12537,NaN,Colwood City Hall -?> 553
8,9,12539,NaN,Courtenay Elementary School -> 526
9,9,12540,NaN,Cranbrook Muriel Baxter -?> 551


In [5]:
def split_station_name(name):
    if pd.isna(name):
        return pd.Series([None, None, 0])

    parts = [p.strip() for p in name.split("->")]

    if len(parts) == 2:
        old_name, new_name = parts
        n_names = 2
    else:
        old_name = new_name = parts[0]
        n_names = 1

    return pd.Series([old_name, new_name, n_names])
    

df_concat_new[['old_name', 'new_name', 'n_names']] = (
    df_concat_new['Unique Names'].apply(split_station_name)
)

df_concat_new = df_concat_new.drop(columns= 'Unique Names')
df_concat_new

,Network ID,Station ID,Native ID,old_name,new_name,n_names
0,9,12939,Fort St John Key Learning Centre -> 469,,Fort St John Key Learning Centre,2.0
1,9,12938,Pine River Gas Plant -> 107,,Pine River Gas Plant,2.0
2,9,2640,M102038 -> 60 -> 294,Harmac Pacific,Harmac Pacific,1.0
3,9,2623,M112070 -> 102,Kelowna College,Kelowna College,1.0
4,9,12531,Birchbank Golf Course -?> 97,NaN,NaN,0.0
5,9,12534,Burns Lake Fire Centre -?> 104,NaN,NaN,0.0
6,9,12535,Burns Lake Sheraton East -?> 439,NaN,NaN,0.0
7,9,12537,Colwood City Hall -?> 553,NaN,NaN,0.0
8,9,12539,Courtenay Elementary School -> 526,NaN,NaN,0.0
9,9,12540,Cranbrook Muriel Baxter -?> 551,NaN,NaN,0.0


In [6]:
import pandas as pd
import re

def parse_native_id(s):
    if pd.isna(s):
        return pd.Series([None, None])

    s = str(s)  # <-- KEY FIX

    parts = re.split(r'\s*-?\?>\s*|\s*->\s*', s)

    old_id = parts[0].strip()
    new_id = parts[-1].strip()

    return pd.Series([old_id, new_id])

df_concat_new[['old_native_id', 'new_native_id']] = df_concat_new['Native ID'].apply(parse_native_id)

df_concat_new = df_concat_new.drop(columns= ['Native ID', 'n_names'])

df_concat_new

,Network ID,Station ID,old_name,new_name,old_native_id,new_native_id
0,9,12939,,Fort St John Key Learning Centre,Fort St John Key Learning Centre,469
1,9,12938,,Pine River Gas Plant,Pine River Gas Plant,107
2,9,2640,Harmac Pacific,Harmac Pacific,M102038,294
3,9,2623,Kelowna College,Kelowna College,M112070,102
4,9,12531,NaN,NaN,Birchbank Golf Course,97
5,9,12534,NaN,NaN,Burns Lake Fire Centre,104
6,9,12535,NaN,NaN,Burns Lake Sheraton East,439
7,9,12537,NaN,NaN,Colwood City Hall,553
8,9,12539,NaN,NaN,Courtenay Elementary School,526
9,9,12540,NaN,NaN,Cranbrook Muriel Baxter,551


In [7]:
select_sql = sa.text("""
SELECT DISTINCT
    n.network_id
FROM meta_station s
JOIN meta_network n
  ON n.network_id = s.network_id
JOIN meta_history m
  ON m.station_id = s.station_id
WHERE s.native_id = :native_id
""")

with engine.begin() as conn:
    for idx, row in df_concat_new.iterrows():

        native_id = row["new_native_id"]

        result = conn.execute(
            select_sql,
            {"native_id": native_id}
        ).fetchall()

        if not result:
            print(f"❌ native_id {native_id} not found")
            continue

        df_concat_new.at[idx, "new_network_id"] = ",".join(
              map(str, {r.network_id for r in result})
        )

df_concat_new

❌ native_id 77 not found
❌ native_id E292249 not found
❌ native_id 0260011 not found


,Network ID,Station ID,old_name,new_name,old_native_id,new_native_id,new_network_id
0,9,12939,,Fort St John Key Learning Centre,Fort St John Key Learning Centre,469,9
1,9,12938,,Pine River Gas Plant,Pine River Gas Plant,107,9
2,9,2640,Harmac Pacific,Harmac Pacific,M102038,294,"9,12"
3,9,2623,Kelowna College,Kelowna College,M112070,102,9
4,9,12531,NaN,NaN,Birchbank Golf Course,97,"9,12"
5,9,12534,NaN,NaN,Burns Lake Fire Centre,104,"9,12"
6,9,12535,NaN,NaN,Burns Lake Sheraton East,439,9
7,9,12537,NaN,NaN,Colwood City Hall,553,9
8,9,12539,NaN,NaN,Courtenay Elementary School,526,"9,12"
9,9,12540,NaN,NaN,Cranbrook Muriel Baxter,551,9


In [8]:
query = """
SELECT
    -- old count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN meta_station s ON m.station_id = s.station_id
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE s.native_id  = %s
       AND s.network_id = %s
    ) AS n_old,

    -- new count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN meta_station s ON m.station_id = s.station_id
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE s.native_id  = %s
       AND s.network_id = %s
    ) AS n_new,

    -- overlap count (time + variable)
    (SELECT COUNT(*)
     FROM (
         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN meta_station s ON m.station_id = s.station_id
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE s.native_id  = %s
           AND s.network_id = %s

         INTERSECT

         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN meta_station s ON m.station_id = s.station_id
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE s.native_id  = %s
           AND s.network_id = %s
     ) t
    ) AS n_overlap,

    -- overlap with identical datum
    (SELECT COUNT(*)
     FROM meta_history m_old
     JOIN meta_station s_old ON m_old.station_id = s_old.station_id
     JOIN obs_raw o_old ON m_old.history_id = o_old.history_id

     JOIN meta_history m_new
       ON 1 = 1
     JOIN meta_station s_new ON m_new.station_id = s_new.station_id
     JOIN obs_raw o_new ON m_new.history_id = o_new.history_id

       AND o_old.obs_time = o_new.obs_time
       AND o_old.vars_id  = o_new.vars_id

     WHERE s_old.native_id  = %s
       AND s_old.network_id = %s
       AND s_new.native_id  = %s
       AND s_new.network_id = %s
       AND ABS(o_old.datum - o_new.datum) < 0.01
    ) AS n_overlap_same_datum;

"""


def count_station_stats(old_id, new_id, old_net, new_net, engine):

    params = (
        old_id, old_net,          # n_old
        new_id, new_net,          # n_new
        old_id, old_net,          # overlap (old)
        new_id, new_net,          # overlap (new)
        old_id, old_net,          # same datum (old)
        new_id, new_net           # same datum (new)
    )

    df = pd.read_sql(
        query,
        engine,
        params=params
    )
    return df.iloc[0]

# df_test = df_concat_new.head(3).copy()

stats = df_concat_new.apply(
    lambda r: count_station_stats(
        r["old_native_id"],
        r["new_native_id"],
        r["Network ID"],
        r["Network ID"],
        engine,
    ),
    axis=1
)

df_concat_new[
    ["n_old", "n_new", "n_overlap", "n_overlap_same_datum"]
] = stats


In [9]:
df_concat_new

,Network ID,Station ID,old_name,new_name,old_native_id,new_native_id,new_network_id,n_old,n_new,n_overlap,n_overlap_same_datum
0,9,12939,,Fort St John Key Learning Centre,Fort St John Key Learning Centre,469,9,0,103326,0,0
1,9,12938,,Pine River Gas Plant,Pine River Gas Plant,107,9,0,640525,0,0
2,9,2640,Harmac Pacific,Harmac Pacific,M102038,294,"9,12",0,626296,0,0
3,9,2623,Kelowna College,Kelowna College,M112070,102,9,0,761324,0,0
4,9,12531,NaN,NaN,Birchbank Golf Course,97,"9,12",0,577611,0,0
5,9,12534,NaN,NaN,Burns Lake Fire Centre,104,"9,12",0,810409,0,0
6,9,12535,NaN,NaN,Burns Lake Sheraton East,439,9,0,296716,0,0
7,9,12537,NaN,NaN,Colwood City Hall,553,9,0,187049,0,0
8,9,12539,NaN,NaN,Courtenay Elementary School,526,"9,12",0,343716,0,0
9,9,12540,NaN,NaN,Cranbrook Muriel Baxter,551,9,0,190393,0,0


In [10]:
df_old_0 = df_concat_new.loc[df_concat_new["n_old"] == 0, ["old_native_id", "Station ID"]]
df_old_0

,old_native_id,Station ID
0,Fort St John Key Learning Centre,12939
1,Pine River Gas Plant,12938
2,M102038,2640
3,M112070,2623
4,Birchbank Golf Course,12531
5,Burns Lake Fire Centre,12534
6,Burns Lake Sheraton East,12535
7,Colwood City Hall,12537
8,Courtenay Elementary School,12539
9,Cranbrook Muriel Baxter,12540


In [11]:
from sqlalchemy import text

exists_sql = text("""
SELECT EXISTS (
    SELECT 1
    FROM meta_history m
    JOIN meta_station s ON m.station_id = s.station_id
    WHERE s.station_id = :station_id
)
""")

exists_list = []

with engine.connect() as conn:   # reuse ONE connection
    for i, row in df_old_0.iterrows():
        station_id = row['Station ID']

        exists = conn.execute(
            exists_sql,
            {"station_id": station_id}
        ).scalar()

        status = "EXISTS" if exists else "NOT EXISTS"
        exists_list.append(status)

        print(
            f"[{i+1:>3}/{len(df_old_0)}] "
            f"station_id={station_id:<15} | "
            f"→ {status}"
        )

# add column to DataFrame
df_old_0['new_station_status'] = exists_list

[  1/52] station_id=12939           | → NOT EXISTS
[  2/52] station_id=12938           | → NOT EXISTS
[  3/52] station_id=2640            | → NOT EXISTS
[  4/52] station_id=2623            | → NOT EXISTS
[  5/52] station_id=12531           | → NOT EXISTS
[  6/52] station_id=12534           | → NOT EXISTS
[  7/52] station_id=12535           | → NOT EXISTS
[  8/52] station_id=12537           | → NOT EXISTS
[  9/52] station_id=12539           | → NOT EXISTS
[ 10/52] station_id=12540           | → NOT EXISTS
[ 11/52] station_id=12542           | → NOT EXISTS
[ 12/52] station_id=12543           | → NOT EXISTS
[ 13/52] station_id=12546           | → NOT EXISTS
[ 14/52] station_id=12547           | → NOT EXISTS
[ 15/52] station_id=12548           | → NOT EXISTS
[ 16/52] station_id=12595           | → NOT EXISTS
[ 17/52] station_id=12550           | → NOT EXISTS
[ 19/52] station_id=12552           | → NOT EXISTS
[ 20/52] station_id=12553           | → NOT EXISTS
[ 21/52] station_id=12596      

### Only concate for the stations belongs to same network 9

In [12]:
df_9 = df_concat_new[
    df_concat_new["new_network_id"].astype(str).str.contains(r"\b9\b")
]

df_not_9 = df_concat_new[
    ~df_concat_new["new_network_id"].astype(str).str.contains(r"\b9\b")
]

In [13]:
df_not_9

,Network ID,Station ID,old_name,new_name,old_native_id,new_native_id,new_network_id,n_old,n_new,n_overlap,n_overlap_same_datum
17,9,12551,NaN,NaN,Horseshoe Bay,T035,72,5646,0,0,0
22,9,12598,NaN,NaN,Kitimat Riverlodge,77,NaN,2756,0,0,0
34,9,12560,NA,PG Marsulex,PG Marsulex,210,12,5657,0,0,0
45,9,12574,NA,Stewart Youth Centre,Stewart Youth Centre,E292249,NaN,0,0,0,0
53,9,12587,NA,Warfield Elementary,Warfield Elementary,0260011,NaN,5660,0,0,0


In [14]:
df_9

,Network ID,Station ID,old_name,new_name,old_native_id,new_native_id,new_network_id,n_old,n_new,n_overlap,n_overlap_same_datum
0,9,12939,,Fort St John Key Learning Centre,Fort St John Key Learning Centre,469,9,0,103326,0,0
1,9,12938,,Pine River Gas Plant,Pine River Gas Plant,107,9,0,640525,0,0
2,9,2640,Harmac Pacific,Harmac Pacific,M102038,294,"9,12",0,626296,0,0
3,9,2623,Kelowna College,Kelowna College,M112070,102,9,0,761324,0,0
4,9,12531,NaN,NaN,Birchbank Golf Course,97,"9,12",0,577611,0,0
5,9,12534,NaN,NaN,Burns Lake Fire Centre,104,"9,12",0,810409,0,0
6,9,12535,NaN,NaN,Burns Lake Sheraton East,439,9,0,296716,0,0
7,9,12537,NaN,NaN,Colwood City Hall,553,9,0,187049,0,0
8,9,12539,NaN,NaN,Courtenay Elementary School,526,"9,12",0,343716,0,0
9,9,12540,NaN,NaN,Cranbrook Muriel Baxter,551,9,0,190393,0,0


In [15]:
SQL_GET_HISTORY = text("""
SELECT h.history_id
FROM meta_history h
JOIN meta_station s ON h.station_id = s.station_id
WHERE s.native_id  = :native_id
  AND s.network_id = :network_id
ORDER BY h.history_id DESC
LIMIT 1
""")

SQL_DELETE_DUPLICATES = text("""
DELETE FROM obs_raw o
USING obs_raw o2
WHERE o.vars_id    = o2.vars_id
  AND o.obs_time   = o2.obs_time
  AND o.history_id = :old_hist_id
  AND o2.history_id = :new_hist_id
""")

SQL_BULK_MOVE = text("""
UPDATE obs_raw
SET history_id = :new_hist_id
WHERE history_id = :old_hist_id
""")


def move_station_obs_fast(old_id, new_id, old_net, new_net, engine):

    with engine.begin() as conn:

        old_hist_id = conn.execute(
            SQL_GET_HISTORY,
            {
                "native_id": old_id,
                "network_id": old_net,
            }
        ).scalar()

        new_hist_id = conn.execute(
            SQL_GET_HISTORY,
            {
                "native_id": new_id,
                "network_id": new_net,
            }
        ).scalar()

        if old_hist_id is None or new_hist_id is None:
            print(
                f"⚠️ Missing history: "
                f"{old_id} (net {old_net}) → {new_id} (net {new_net})"
            )
            return 0

        # 1️⃣ delete duplicates first
        dup_res = conn.execute(
            SQL_DELETE_DUPLICATES,
            {
                "old_hist_id": old_hist_id,
                "new_hist_id": new_hist_id,
            }
        )

        # 2️⃣ bulk reassign
        move_res = conn.execute(
            SQL_BULK_MOVE,
            {
                "old_hist_id": old_hist_id,
                "new_hist_id": new_hist_id,
            }
        )

        print(
            f"🧹 removed duplicates={dup_res.rowcount}, "
            f"🚚 moved={move_res.rowcount} "
            f"({old_id}/{old_net} → {new_id}/{new_net})"
        )

        return move_res.rowcount


results = []

for i, row in df_9.iterrows():
    old_id  = row["old_native_id"]
    new_id  = row["new_native_id"]
    old_net = row["Network ID"]
    new_net = row["Network ID"]

    print(
        f"[{i+1}/{len(df_9)}] "
        f"{old_id} (net {old_net}) → {new_id} (net {new_net})"
    )

    n = move_station_obs_fast(
        old_id,
        new_id,
        old_net,
        new_net,
        engine,
    )

    results.append(n)

print("All done!")

[1/51] Fort St John Key Learning Centre (net 9) → 469 (net 9)
⚠️ Missing history: Fort St John Key Learning Centre (net 9) → 469 (net 9)
[2/51] Pine River Gas Plant (net 9) → 107 (net 9)
⚠️ Missing history: Pine River Gas Plant (net 9) → 107 (net 9)
[3/51] M102038 (net 9) → 294 (net 9)
⚠️ Missing history: M102038 (net 9) → 294 (net 9)
[4/51] M112070 (net 9) → 102 (net 9)
⚠️ Missing history: M112070 (net 9) → 102 (net 9)
[5/51] Birchbank Golf Course (net 9) → 97 (net 9)
⚠️ Missing history: Birchbank Golf Course (net 9) → 97 (net 9)
[6/51] Burns Lake Fire Centre (net 9) → 104 (net 9)
⚠️ Missing history: Burns Lake Fire Centre (net 9) → 104 (net 9)
[7/51] Burns Lake Sheraton East (net 9) → 439 (net 9)
⚠️ Missing history: Burns Lake Sheraton East (net 9) → 439 (net 9)
[8/51] Colwood City Hall (net 9) → 553 (net 9)
⚠️ Missing history: Colwood City Hall (net 9) → 553 (net 9)
[9/51] Courtenay Elementary School (net 9) → 526 (net 9)
⚠️ Missing history: Courtenay Elementary School (net 9) → 526

In [ ]:
# SQL_NEW_HISTORY = text("""
# SELECT h.history_id
# FROM meta_history h
# JOIN meta_station s ON h.station_id = s.station_id
# WHERE s.native_id = :new_id
#   AND s.network_id = :new_network_id
# ORDER BY h.history_id DESC
# LIMIT 1
# """)


# SQL_MOVE_UPDATE = text("""
# WITH updated AS (
#     UPDATE obs_raw o
#     SET history_id = :new_hist_id
#     FROM meta_history h_old
#     JOIN meta_station s_old
#       ON h_old.station_id = s_old.station_id
#     WHERE o.history_id = h_old.history_id
#       AND s_old.native_id  = :old_id
#       AND s_old.network_id = :old_network_id
#       AND NOT EXISTS (
#           SELECT 1
#           FROM obs_raw o_new
#           JOIN meta_history h_new
#             ON o_new.history_id = h_new.history_id
#           JOIN meta_station s_new
#             ON h_new.station_id = s_new.station_id
#           WHERE s_new.native_id  = :new_id
#             AND s_new.network_id = :new_network_id
#             AND o_new.obs_time = o.obs_time
#             AND o_new.vars_id  = o.vars_id
#       )
#     RETURNING 1
# )
# SELECT COUNT(*) FROM updated;
# """)


# def move_station_obs_fast(old_id, new_id, old_net, new_net, engine):

#     with engine.begin() as conn:

#         # 1. get new history_id
#         new_hist_id = conn.execute(
#             SQL_NEW_HISTORY,
#             {
#                 "new_id": new_id,
#                 "new_network_id": new_net,
#             }
#         ).scalar()

#         if new_hist_id is None:
#             print(
#                 f"⚠️ New station '{new_id}' (network {new_net}) "
#                 f"has no history, skipping."
#             )
#             return 0

#         # 2. move observations
#         n_move = conn.execute(
#             SQL_MOVE_UPDATE,
#             {
#                 "old_id": old_id,
#                 "new_id": new_id,
#                 "old_network_id": old_net,
#                 "new_network_id": new_net,
#                 "new_hist_id": new_hist_id,
#             }
#         ).scalar()

#         print(
#             f"✅ Moved {n_move} rows "
#             f"from '{old_id}' (net {old_net}) "
#             f"→ '{new_id}' (net {new_net})"
#         )

#         return n_move


# results = []

# for i, row in df_9.iterrows():
#     old_id  = row["old_native_id"]
#     new_id  = row["new_native_id"]
#     old_net = row["Network ID"]
#     new_net = row["Network ID"]

#     print(
#         f"[{i+1}/{len(df_9)}] "
#         f"{old_id} (net {old_net}) → {new_id} (net {new_net})"
#     )

#     n = move_station_obs_fast(
#         old_id,
#         new_id,
#         old_net,
#         new_net,
#         engine,
#     )

#     results.append(n)

# # df_9["n_moved"] = results
# print("All done!")

[1/52] Fort St John Key Learning Centre (net 9) → 469 (net 9)
✅ Moved 981 rows from 'Fort St John Key Learning Centre' (net 9) → '469' (net 9)
[2/52] Pine River Gas Plant (net 9) → 107 (net 9)
✅ Moved 502 rows from 'Pine River Gas Plant' (net 9) → '107' (net 9)
[3/52] M102038 (net 9) → 294 (net 9)
✅ Moved 563442 rows from 'M102038' (net 9) → '294' (net 9)
[4/52] M112070 (net 9) → 102 (net 9)
✅ Moved 469068 rows from 'M112070' (net 9) → '102' (net 9)
[5/52] Birchbank Golf Course (net 9) → 97 (net 9)
✅ Moved 4496 rows from 'Birchbank Golf Course' (net 9) → '97' (net 9)
[6/52] Burns Lake Fire Centre (net 9) → 104 (net 9)
✅ Moved 4512 rows from 'Burns Lake Fire Centre' (net 9) → '104' (net 9)
[7/52] Burns Lake Sheraton East (net 9) → 439 (net 9)
✅ Moved 4509 rows from 'Burns Lake Sheraton East' (net 9) → '439' (net 9)
[8/52] Colwood City Hall (net 9) → 553 (net 9)
✅ Moved 4010 rows from 'Colwood City Hall' (net 9) → '553' (net 9)
[9/52] Courtenay Elementary School (net 9) → 526 (net 9)
✅ M

### update the station name

In [21]:
select_sql = sa.text("""
SELECT DISTINCT
    m.station_name
FROM meta_station s
JOIN meta_history m
  ON m.station_id = s.station_id
WHERE s.native_id  = :native_id
  AND s.network_id = :network_id
""")

with engine.begin() as conn:
    for idx, row in df_9.iterrows():

        native_id = row["new_native_id"]
        network_id = row["Network ID"]

        result = conn.execute(
            select_sql,
            {"native_id": native_id,
            "network_id": network_id}
        ).fetchall()

        if not result:
            print(f"❌ native_id {native_id} not found")
            continue

        df_9.at[idx, "new_station_name_db"] = ",".join(
              map(str, {r.station_name for r in result})
        )

df_9

❌ native_id E292249 not found


,Network ID,Station ID,old_name,new_name,old_native_id,new_native_id,new_network_id,n_old,n_new,n_overlap,n_overlap_same_datum,new_station_name_db
0,9,12939,,Fort St John Key Learning Centre,Fort St John Key Learning Centre,469,9,2131,102345,1150,1150,Fort St John Key Learning Centre
1,9,12938,,Pine River Gas Plant,Pine River Gas Plant,107,9,502,640023,0,0,Pine River Gas Plant
2,9,2640,Harmac Pacific,Harmac Pacific,M102038,294,"9,12",563442,60598,0,0,NaN
3,9,2623,Kelowna College,Kelowna College,M112070,102,9,473357,290713,4289,294,Kelowna College
4,9,12531,NaN,NaN,Birchbank Golf Course,97,"9,12",5640,573115,1144,1144,NaN
5,9,12534,NaN,NaN,Burns Lake Fire Centre,104,"9,12",5660,805897,1148,1148,NaN
6,9,12535,NaN,NaN,Burns Lake Sheraton East,439,9,5661,292207,1152,1152,NaN
7,9,12537,NaN,NaN,Colwood City Hall,553,"9,12",5154,183039,1144,1144,NaN
8,9,12539,NaN,NaN,Courtenay Elementary School,526,"9,12",5637,339207,1128,1128,NaN
9,9,12540,NaN,NaN,Cranbrook Muriel Baxter,551,"9,12",5656,185883,1146,1146,NaN


In [18]:
update_sql = sa.text("""
UPDATE meta_history h
SET station_name = :new_name
FROM meta_station s
WHERE h.station_id = s.station_id
  AND s.native_id  = :native_id
  AND s.network_id = :network_id
""")

with engine.begin() as conn:
    for _, row in df_9.iterrows():

        native_id  = row["new_native_id"]
        network_id = row["Network ID"]
        new_name   = row["new_name"]

        result = conn.execute(
            update_sql,
            {
                "new_name": new_name,
                "native_id": native_id,
                "network_id": network_id
            }
        )

        if result.rowcount >= 1:
            print(
                f"✅ Updated station '{native_id}' "
                f"(network {network_id}) → '{new_name}' "
                f"({result.rowcount} row(s) updated)"
            )
        else:
            print(f"⚠️ Station '{native_id}' (network {network_id}): no update applied")

✅ Updated station '469' (network 9) → 'Fort St John Key Learning Centre' (1 row(s) updated)
✅ Updated station '107' (network 9) → 'Pine River Gas Plant' (1 row(s) updated)
✅ Updated station '294' (network 9) → 'Harmac Pacific' (1 row(s) updated)
✅ Updated station '102' (network 9) → 'Kelowna College' (1 row(s) updated)
✅ Updated station '97' (network 9) → 'nan' (1 row(s) updated)
✅ Updated station '104' (network 9) → 'nan' (1 row(s) updated)
✅ Updated station '439' (network 9) → 'nan' (1 row(s) updated)
✅ Updated station '553' (network 9) → 'nan' (1 row(s) updated)
✅ Updated station '526' (network 9) → 'nan' (1 row(s) updated)
✅ Updated station '551' (network 9) → 'nan' (1 row(s) updated)
✅ Updated station '39' (network 9) → 'nan' (1 row(s) updated)
✅ Updated station '537' (network 9) → 'nan' (1 row(s) updated)
✅ Updated station '452' (network 9) → 'nan' (2 row(s) updated)
✅ Updated station '19' (network 9) → 'nan' (1 row(s) updated)
✅ Updated station '56' (network 9) → 'nan' (1 row(s)

### Delete the old stations

In [23]:
from tqdm import tqdm
import sqlalchemy as sa

station_ids = (
    df_9["Station ID"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)
# station_ids[0]

# List of station_ids to delete
station_ids_to_delete = station_ids  # or a subset

delete_obs_flags = sa.text("""
DELETE FROM obs_raw_pcic_flags fr
USING obs_raw o, meta_history h
WHERE fr.obs_raw_id = o.obs_raw_id
  AND o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs = sa.text("""
DELETE FROM obs_raw o
USING meta_history h
WHERE o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs_derived = sa.text("""
DELETE FROM obs_derived_values dv
USING meta_history h
WHERE dv.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_history = sa.text("""
DELETE FROM meta_history
WHERE station_id = :station_id
""")

delete_station = sa.text("""
DELETE FROM meta_station
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for sid in tqdm(station_ids_to_delete, desc="Deleting stations"):

        # 1️⃣ obs_raw_pcic_flags (depends on obs_raw)
        res_flags = conn.execute(
            delete_obs_flags,
            {"station_id": sid}
        )

        # 2️⃣ obs_raw
        res_obs = conn.execute(
            delete_obs,
            {"station_id": sid}
        )

        # 3️⃣ obs_derived_values (depends on meta_history)
        res_obs_dv = conn.execute(
            delete_obs_derived,
            {"station_id": sid}
        )

        # 4️⃣ meta_history
        res_hist = conn.execute(
            delete_history,
            {"station_id": sid}
        )

        # 5️⃣ meta_station
        res_sta = conn.execute(
            delete_station,
            {"station_id": sid}
        )

        tqdm.write(
            f"Station {sid}: "
            f"flags={res_flags.rowcount}, "
            f"obs_raw={res_obs.rowcount}, "
            f"obs_derived={res_obs_dv.rowcount}, "
            f"meta_history={res_hist.rowcount}, "
            f"meta_station={res_sta.rowcount}"
        )

Deleting stations:  79%|███████▉  | 41/52 [00:00<00:00, 204.93it/s]

Station 12939: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 12938: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 2640: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 2623: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 12531: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 12534: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 12535: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 12537: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 12539: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 12540: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 12542: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 12543: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 12546: flags=0, obs_ra

Deleting stations: 100%|██████████| 52/52 [00:00<00:00, 209.66it/s]

Station 12574: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 12577: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 12578: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 12579: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 12581: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 12584: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 12585: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 12586: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 12588: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
Station 2618: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=0
